# Phase 5 diagnostic (inference only): over-enhancement across exposure levels

Upload to Drive `MyDrive/LowLight-Generalization/phase5_inputs/`: `phase5_diag_bundle_20261108_1823.zip`, `phase5_diag_weights_20261108_1823.zip`, `phase5_diag_data.zip`.
Runtime: **T4 GPU**. Run cells 1-6 in order in one session. Outputs go to
`experiments/phase5_llie_direction/diagnostic/` on Drive. Reruns skip finished (dataset, condition) parts.

No training, no parameter updates, **no SICE Part 2** (it is not in the data pack and the script refuses it).

In [ ]:
# 1. GPU
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
print(gpu, torch.__version__)
assert 'T4' in gpu, 'expected a Tesla T4 runtime: STOP'

In [ ]:
# 2. Drive, integrity of the three inputs, extraction
from google.colab import drive; drive.mount('/content/drive')
import glob, hashlib, json, os, subprocess, sys, time, zipfile
DRIVE = '/content/drive/MyDrive/LowLight-Generalization'
IN = f'{DRIVE}/phase5_inputs'
OUT = f'{DRIVE}/experiments/phase5_llie_direction/diagnostic'
os.makedirs(f'{OUT}/session_records', exist_ok=True); os.makedirs(f'{OUT}/logs', exist_ok=True)
def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
for name, h in (('phase5_diag_bundle_20261108_1823.zip', '3ad755670b488057d57c282ec62eacaa'), ('phase5_diag_weights_20261108_1823.zip', 'bf9c4e0a6940fc6c81e101385545d731'), ('phase5_diag_data.zip', '406a5d343c877a7b1b3456d1224cee0b')):
    assert md5(f'{IN}/{name}') == h, f'{name} MD5 mismatch: STOP'
    print('OK', name)
PROJ, W, DATA = '/content/p5proj', '/content/p5weights', '/content/p5data'
zipfile.ZipFile(f'{IN}/phase5_diag_bundle_20261108_1823.zip').extractall(PROJ)
man = json.load(open(f'{PROJ}/BUNDLE_MANIFEST.json'))['files']
assert not [f for f, h in man.items() if md5(f'{PROJ}/{f}') != h], 'bundle file MD5 mismatch: STOP'
if not os.path.isdir(W): zipfile.ZipFile(f'{IN}/phase5_diag_weights_20261108_1823.zip').extractall(W)
if not os.path.isdir(DATA): zipfile.ZipFile(f'{IN}/phase5_diag_data.zip').extractall(DATA)
dm = json.load(open(f'{DATA}/manifest.json'))
bad = [f for f, h in dm['files'].items() if md5(f'{DATA}/{f}') != h]
assert not bad, f'data file MD5 mismatch: {bad[:3]}: STOP'
assert not glob.glob(f'{DATA}/**/*Part2*', recursive=True), 'SICE Part 2 found in the data pack: STOP'
print('bundle', len(man), 'files; data', dm['counts'])
STAMP = time.strftime('%Y%m%d_%H%M%S')

In [ ]:
# 3. Packages: lpips (as Phase 2) and pyiqa 0.1.16 for NIQE / BRISQUE, with core packages pinned to this session's versions
import subprocess, sys
def freeze():
    return dict(l.split('==', 1) for l in subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True,
                text=True).stdout.splitlines() if '==' in l)
before = freeze()
pins = {k: v for k, v in before.items() if k.lower() in ['torch', 'torchvision', 'torchaudio', 'numpy', 'pillow', 'scipy', 'opencv-python', 'opencv-python-headless', 'lpips']}
open('/content/constraints.txt', 'w').write('\n'.join(f'{k}=={v}' for k, v in pins.items()))
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lpips==0.1.4', 'pyiqa==0.1.16', '-c', '/content/constraints.txt'],
                   capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-3000:])
assert r.returncode == 0, 'pip could not install lpips/pyiqa without changing pinned packages: STOP (paste this output)'
after = freeze()
changed = {k: [before.get(k), after.get(k)] for k in set(before) | set(after) if before.get(k) != after.get(k)}
assert all(before.get(k) == after.get(k) for k in pins), 'a pinned core package changed: STOP'
rec = {'pinned_unchanged': pins, 'installed_or_changed': changed, 'python': sys.version}
json.dump(rec, open(f'{OUT}/session_records/packages_{STAMP}.json', 'w'), indent=1)
open(f'{OUT}/session_records/pip_freeze_{STAMP}.txt', 'w').write('\n'.join(f'{k}=={v}' for k, v in sorted(after.items())))
print('changed packages:', changed)

In [ ]:
# 4. Stage runner (logs to Drive; stops on any failure)
def run_stage(stage, *extra):
    log = f'{OUT}/logs/diag_{stage}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    cmd = [sys.executable, 'scripts/phase5_diagnostic.py', stage, '--data', DATA, '--weights', W, '--out', OUT, *extra]
    with open(log, 'w') as f:
        p = subprocess.Popen(cmd, cwd=PROJ, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end=''); f.write(line); f.flush()
        rc = p.wait()
    assert rc == 0, f'stage {stage} FAILED (exit {rc}); log: {log}. STOP'
    print(f'stage {stage} OK; log: {log}')

In [ ]:
# 5. Inference on all datasets x conditions (resumable; about 1.5-3 h on a T4, estimate)
run_stage('run')

In [ ]:
# 6. Report: tables, plots, panels (CPU; a few minutes)
run_stage('report')
print(open(f'{OUT}/DIAG_REPORT.md').read()[:20000])